# 11. 확인 실험: 새 시드 10개로 D(의사라벨) 고정 여부 판단
10에서 쓴 시드(0~4)와 겹치지 않는 새 시드 5~14로 다시 돌린다(결과에 맞춘 선택을 피하려는 확인 실험).
**판단 규칙(사전 고정)**: D가 4개 조건 평균 같은 위치 안 AUROC에서 A·C 이상이고, 원본·8묶음 4% 조건에서 기준 2·3을 다시 통과하면 최종 약지도 모델로 고정한다.

In [1]:
import sys, time, numpy as np, pandas as pd, torch
sys.path.insert(0, ".")
from joblib import Parallel, delayed
from wl import runner, bags
pd.set_option("display.width", 250)
MODELS = ["const_rate", "A_dllp_mlp", "C_prior_two", "D_pseudo_logreg", "UB_logreg"]
CONDS = list(bags.CONDITIONS); SEEDS = list(range(5, 15)); N_GPU = torch.cuda.device_count()
name = "shape_confirm"
def _task(i, m, c, s): return runner.run_one(m, c, s, feature_set="shape_locstd", gpu=i % max(N_GPU, 1))
if (runner.RES / f"{name}.pkl").exists():
    runs = runner.load_runs(name)
else:
    tasks = [(m, c, s) for m in MODELS for c in CONDS for s in SEEDS]
    t = time.time(); runs = Parallel(n_jobs=8, verbose=2)(delayed(_task)(i, *k) for i, k in enumerate(tasks))
    print(f"{len(runs)}개 실행 {time.time()-t:.0f}초"); runner.save_runs(runs, name)
by = {}
for r in runs: by.setdefault((r["model"], r["cond"]), []).append(r)
boots = dict(Parallel(n_jobs=16)(delayed(lambda k: (k, runner.bootstrap(by[k], n_boot=200)))(k) for k in by))
def fmt(m, c, k):
    r = boots[(m, c)].loc[k]; return f"{r['mean']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}]"
KEYS = ["auroc", "wl_auroc", "recall@0.1", "wl_recall@0.1", "recall@5x"]
for c in CONDS:
    print(f"\n=== shape_locstd | {c} ==="); display(pd.DataFrame({k: {m: fmt(m, c, k) for m in MODELS} for k in KEYS}))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    2.5s


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[Parallel(n_jobs=8)]: Done 162 tasks      | elapsed:  3.3min


[Parallel(n_jobs=8)]: Done 200 out of 200 | elapsed:  3.4min finished


200개 실행 205초


... (4 outputs left out)


=== shape_locstd | k8_1 ===


,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.048 [0.046, 0.050]"
A_dllp_mlp,"0.592 [0.552, 0.636]","0.596 [0.538, 0.657]","0.147 [0.094, 0.222]","0.139 [0.083, 0.175]","0.102 [0.042, 0.195]"
C_prior_two,"0.474 [0.442, 0.531]","0.466 [0.411, 0.534]","0.111 [0.072, 0.151]","0.104 [0.065, 0.143]","0.084 [0.047, 0.132]"
D_pseudo_logreg,"0.540 [0.502, 0.588]","0.552 [0.475, 0.623]","0.079 [0.040, 0.133]","0.117 [0.070, 0.180]","0.059 [0.019, 0.108]"
UB_logreg,"0.608 [0.556, 0.654]","0.602 [0.533, 0.682]","0.274 [0.157, 0.361]","0.157 [0.106, 0.240]","0.161 [0.076, 0.232]"



=== shape_locstd | k8_05 ===


,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.025 [0.023, 0.026]"
A_dllp_mlp,"0.577 [0.518, 0.634]","0.562 [0.448, 0.649]","0.090 [0.033, 0.170]","0.097 [0.060, 0.177]","0.072 [0.017, 0.133]"
C_prior_two,"0.471 [0.435, 0.519]","0.445 [0.376, 0.529]","0.080 [0.029, 0.167]","0.064 [0.029, 0.141]","0.053 [0.018, 0.110]"
D_pseudo_logreg,"0.503 [0.451, 0.561]","0.506 [0.442, 0.571]","0.100 [0.039, 0.169]","0.094 [0.056, 0.148]","0.020 [0.000, 0.050]"
UB_logreg,"0.622 [0.538, 0.697]","0.631 [0.487, 0.739]","0.270 [0.092, 0.397]","0.191 [0.103, 0.274]","0.122 [0.028, 0.270]"


In [2]:
rows = []
for m in MODELS[1:]:
    row = {"모델": m}
    for c in CONDS:
        b = boots[(m, c)]
        row[c] = ("O" if b.loc["recall@0.1", "lo"] > .10 else "X") + ("O" if (b.loc["wl_recall@0.1", "lo"] > .10 and b.loc["wl_auroc", "lo"] > .5) else "X")
    rows.append(row)
print("[기준2 포착률][기준3 같은 위치 안]"); pd.DataFrame(rows)

[기준2 포착률][기준3 같은 위치 안]


,모델,orig,k8_4,k8_1,k8_05
0,A_dllp_mlp,OO,OO,XX,XX
1,C_prior_two,OO,OX,XX,XX
2,D_pseudo_logreg,OO,XO,XX,XX
3,UB_logreg,OO,OO,OO,XX


In [3]:
avg = {m: np.mean([boots[(m, c)].loc["wl_auroc", "mean"] for c in CONDS]) for m in MODELS[1:]}
print("4조건 평균 같은 위치 안 AUROC:", {m: round(v, 3) for m, v in avg.items()})

4조건 평균 같은 위치 안 AUROC: {'A_dllp_mlp': np.float64(0.612), 'C_prior_two': np.float64(0.552), 'D_pseudo_logreg': np.float64(0.601), 'UB_logreg': np.float64(0.648)}
